In [1]:

!
!pip uninstall -y datasets
!pip install -q datasets==3.6.0 pandas pyarrow sentence-transformers transformers scikit-learn nltk tqdm faiss-cpu


Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 83.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0

In [2]:
import datasets

print(datasets.__version__)

3.6.0


In [3]:
from datasets import load_dataset

dataset_dict = load_dataset(
    "HUPD/hupd",
    name="sample",
    data_files="https://huggingface.co/datasets/HUPD/hupd/blob/main/hupd_metadata_2022-02-22.feather",
    icpr_label=None,
    train_filing_start_date="2016-01-01",
    train_filing_end_date="2016-01-21",
    val_filing_start_date="2016-01-22",
    val_filing_end_date="2016-01-31",
)

README.md:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

hupd.py:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

The repository for HUPD/hupd contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/HUPD/hupd.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Loading dataset with config: PatentsConfig(name='sample', version=0.0.0, data_dir='sample', data_files={'train': ['https://huggingface.co/datasets/HUPD/hupd/blob/main/hupd_metadata_2022-02-22.feather']}, description='Patent data from January 2016, for debugging')


hupd_metadata_jan16_2022-02-22.feather: reconstructing file:   0%|          |  0.00B / 6.67MB            

hupd_metadata_jan16_2022-02-22.feather: downloading bytes:           |  0.00B            

... (2 outputs left out)

data/sample-jan-2016.tar.gz: downloading bytes:           |  0.00B            

Reading metadata file: /root/.cache/huggingface/hub/datasets--HUPD--hupd/snapshots/f570a84b03663180b6034c1f7f4c15864f94385e/hupd_metadata_jan16_2022-02-22.feather
Filtering train dataset by filing start date: 2016-01-01
Filtering train dataset by filing end date: 2016-01-21
Filtering val dataset by filing start date: 2016-01-22
Filtering val dataset by filing end date: 2016-01-31


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [4]:
import os
import re
import gc
import random
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
SEED = 2024

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Random seed: 2024


In [5]:
df = dataset_dict["train"].to_pandas()

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (16153, 14)
['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id']


In [6]:
rename_map = {}

if "patent_number" in df.columns:
    rename_map["patent_number"] = "patent_id"

if "cpc_label" in df.columns:
    rename_map["cpc_label"] = "cpc"

if "ipc_label" in df.columns:
    rename_map["ipc_label"] = "ipc"

df = df.rename(columns=rename_map)

print(df.columns.tolist())

['patent_id', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc', 'ipc', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id']


In [7]:
df

,patent_id,decision,title,abstract,claims,background,summary,description,cpc,ipc,filing_date,patent_issue_date,date_published,examiner_id
0,13261748,ACCEPTED,MINI-OPTICAL NETWORK TERMINAL (ONT),The present invention relates to passive optic...,"1. A compact optical network terminal, compris...",<SOH> BACKGROUND OF THE INVENTION <EOH>A netwo...,<SOH> SUMMARY OF THE INVENTION <EOH>An aspect ...,FIELD OF THE INVENTION The present invention r...,H04Q110071,H04Q1100,20160120,20170606,20160526,95191.0
1,13995128,ACCEPTED,APPARATUS FOR FORMING AND READING AN IDENTIFIC...,Embodiments of the invention provide a method ...,1. A method comprising: using a first reader t...,<SOH> BACKGROUND OF THE INVENTION <EOH>Identif...,<SOH> SUMMARY OF THE INVENTION <EOH>In accorda...,CROSS-REFERENCE TO RELATED APPLICATIONS The pr...,G06K500,G06K500,20160112,20160322,20140102,59514.0
2,14241799,PENDING,PORTABLE DRUG DISPENSER,A portable drug dispenser includes a chamber f...,"1. A portable drug dispenser, comprising: a ch...",,,This application claims priority from U.S. app...,A61J70084,A61J700,20160104,,20171116,95928.0
3,14348792,ACCEPTED,LIQUID-COOLED HEAT EXCHANGER,A crystal growth furnace comprising a crucible...,1. A crystal growth furnace for growing a crys...,<SOH> BACKGROUND OF THE INVENTION <EOH>1. Fiel...,<SOH> SUMMARY OF THE INVENTION <EOH>The presen...,CROSS-REFERENCE TO RELATED APPLICATIONS The pr...,C30B11003,C30B1100,20160111,20180529,20160512,63013.0
4,14360978,REJECTED,SOLE MEMBER OF FOOTWEAR,A shoe midsole is composed of a base plate (1)...,1. A sole member of footwear comprising a base...,<SOH> BACKGROUND ART <EOH>When the heel touche...,<SOH> BRIEF DESCRIPTION OF THE DRAWINGS <EOH>F...,TECHNICAL FIELD The present invention relates ...,A43B13181,A43B1318,20160113,,20160512,94490.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16148,15002394,ACCEPTED,ROBOT HAND CONTROLLING METHOD AND ROBOTICS DEVICE,A robot hand controlling method executes calcu...,"1. A controlling method of a robot hand, the r...",<SOH> BACKGROUND OF THE INVENTION <EOH>1. Fiel...,<SOH> SUMMARY OF THE INVENTION <EOH>An object ...,BACKGROUND OF THE INVENTION 1. Field of the In...,B25J91612,B25J916,20160120,20180710,20160804,66148.0
16149,15002396,REJECTED,IMMUNOGLOBULIN FUSION PROTEINS AND USES THEREOF,A fusion protein is disclosed. The fusion prot...,1. A fusion protein comprising an Fc fragment ...,<SOH> BACKGROUND OF THE INVENTION <EOH>An immu...,<SOH> SUMMARY OF THE INVENTION <EOH>The presen...,The present application is a U.S. Nonprovision...,C07K14745,C07K14745,20160120,,20161215,95819.0
16150,15330955,REJECTED,PIPE EXTRACTION TOOL,A pipe extraction tool that grips the inside o...,1. A pipe extraction tool for extracting a pip...,<SOH> BACKGROUND OF THE INVENTION <EOH>1. Fiel...,<SOH> BRIEF SUMMARY OF THE INVENTION <EOH>The ...,CROSS-REFERENCES TO RELATED APPLICATIONS Not a...,B25B2714,B25B2714,20160120,,20170907,95661.0
16151,15330961,PENDING,Molded parts with thermoplastic cellulose biop...,A longitudinal extending body with oriented fi...,1. A longitudinal body of a solidified organic...,<SOH> BACKGROUND OF INVENTION <EOH>In the medi...,<SOH> BRIEF SUMMARY OF THE PRESENT INVENTION <...,CROSS REFERENCES Application claims priority o...,A61L3106,A61L3106,20160111,,20171019,96956.0


In [8]:
def clean_text(text):

    if text is None:
        return ""

    text = str(text)

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    text = text.replace("\n", " ")
    text = text.replace("\r", " ")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [9]:
def normalize_classification(code):

    if code is None:
        return ""

    code = str(code).upper().strip()

    code = re.sub(
        r"[^A-Z0-9]",
        "",
        code
    )

    return code

In [10]:
required_text_columns = [
    "title",
    "abstract",
    "claims"
]

required_code_columns = [
    "ipc",
    "cpc"
]

for col in required_text_columns + required_code_columns:

    if col not in df.columns:
        df[col] = ""


for col in required_text_columns:

    df[col] = (
        df[col]
        .fillna("")
        .apply(clean_text)
    )


for col in required_code_columns:

    df[col] = (
        df[col]
        .fillna("")
        .apply(normalize_classification)
    )

In [11]:
df = df.drop_duplicates(
    subset=["patent_id"]
).reset_index(drop=True)


df = df[
    (df["abstract"].str.len() > 30) &
    (df["claims"].str.len() > 30)
].reset_index(drop=True)


print("After cleaning:", df.shape)

After cleaning: (16153, 14)


In [12]:
MAX_PATENTS = 10000

if len(df) > MAX_PATENTS:

    df = df.sample(
        n=MAX_PATENTS,
        random_state=SEED
    ).reset_index(drop=True)


print("Number of patents:", len(df))

Number of patents: 10000


In [13]:
def create_model_text(row):

    title = row["title"][:500]

    abstract = row["abstract"][:3000]

    claims = row["claims"][:3000]

    return (
        "Title: " + title +
        " Abstract: " + abstract +
        " Claims: " + claims
    )


df["model_text"] = df.apply(
    create_model_text,
    axis=1
)

In [14]:
df = df[
    [
        "patent_id",
        "title",
        "abstract",
        "claims",
        "model_text",
        "ipc",
        "cpc"
    ]
].copy()

print(df.shape)

(10000, 7)


In [15]:
from sklearn.model_selection import train_test_split

all_indices = np.arange(
    len(df)
)


train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=SEED
)


val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED
)


print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Train: 8000
Validation: 1000
Test: 1000


In [16]:
def subclass_code(code):

    code = normalize_classification(code)

    if len(code) >= 4:
        return code[:4]

    return code


df["ipc_subclass"] = df["ipc"].apply(
    subclass_code
)

df["cpc_subclass"] = df["cpc"].apply(
    subclass_code
)

In [17]:
def create_training_pairs(
    df,
    indices,
    positive_pairs=4000,
    negative_pairs=4000
):

    indices = list(indices)

    ipc_groups = {}
    cpc_groups = {}

    for idx in indices:

        ipc = df.iloc[idx]["ipc_subclass"]
        cpc = df.iloc[idx]["cpc_subclass"]

        if ipc:
            ipc_groups.setdefault(
                ipc,
                []
            ).append(idx)

        if cpc:
            cpc_groups.setdefault(
                cpc,
                []
            ).append(idx)


    pairs = []


    # -------------------------
    # Positive pairs
    # -------------------------

    for _ in range(positive_pairs):

        use_ipc = random.random() < 0.5

        groups = (
            ipc_groups
            if use_ipc
            else cpc_groups
        )

        valid_groups = [
            g
            for g, members in groups.items()
            if len(members) >= 2
        ]

        if not valid_groups:
            continue

        group = random.choice(
            valid_groups
        )

        a, b = random.sample(
            groups[group],
            2
        )

        pairs.append(
            (a, b, 1.0)
        )


    # -------------------------
    # Negative pairs
    # -------------------------

    attempts = 0

    while (
        len(pairs)
        < positive_pairs + negative_pairs
        and attempts < negative_pairs * 10
    ):

        attempts += 1

        a, b = random.sample(
            indices,
            2
        )

        ipc_a = df.iloc[a]["ipc_subclass"]
        ipc_b = df.iloc[b]["ipc_subclass"]

        cpc_a = df.iloc[a]["cpc_subclass"]
        cpc_b = df.iloc[b]["cpc_subclass"]

        if (
            ipc_a != ipc_b
            and
            cpc_a != cpc_b
        ):

            pairs.append(
                (a, b, 0.0)
            )


    random.shuffle(pairs)

    return pairs

In [18]:
train_pairs = create_training_pairs(
    df,
    train_idx,
    positive_pairs=4000,
    negative_pairs=4000
)

print(
    "Training pairs:",
    len(train_pairs)
)

Training pairs: 8000


In [19]:
class PatentPairDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        df,
        pairs
    ):

        self.df = df
        self.pairs = pairs


    def __len__(self):

        return len(self.pairs)


    def __getitem__(
        self,
        index
    ):

        a, b, label = self.pairs[index]

        text_a = self.df.iloc[a][
            "model_text"
        ]

        text_b = self.df.iloc[b][
            "model_text"
        ]

        return InputExample(
            texts=[
                text_a,
                text_b
            ],
            label=float(label)
        )

In [20]:

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses
)
model = SentenceTransformer(
    "AI-Growth-Lab/PatentSBERTa"
)

model.max_seq_length = 256

print(
    "Model loaded."
)

print(
    "Max sequence length:",
    model.max_seq_length
)

/tmp/ipykernel_49/3007173248.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.88k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.
Max sequence length: 256


In [21]:
texts = df["model_text"].tolist()

pretrained_embeddings = model.encode(
    texts,
    batch_size=2,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

pretrained_embeddings = (
    pretrained_embeddings
    .astype(np.float32)
)

del texts

gc.collect()

print(
    "Shape:",
    pretrained_embeddings.shape
)

print(
    "Memory:",
    pretrained_embeddings.nbytes /
    (1024 ** 2),
    "MB"
)

Batches:   0%|          | 0/5000 [00:00<?, ?it/s]

Shape: (10000, 768)
Memory: 29.296875 MB


In [22]:
from torch.utils.data import DataLoader
import torch
train_dataset = PatentPairDataset(
    df,
    train_pairs
)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    collate_fn=model.smart_batching_collate
)

In [23]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

import random
import numpy as np
import torch

SEED = 2024
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed:", SEED)

Seed: 2024


In [24]:
# ============================================================
# VALIDATION DATA
# ============================================================

val_df = df.iloc[val_idx].reset_index(drop=True)

print("Validation patents:", len(val_df))

Validation patents: 1000


In [25]:
val_df = df.iloc[val_idx].reset_index(drop=True)

print("Validation patents:", len(val_df))
print(val_df[["patent_id", "ipc", "cpc"]].head())

Validation patents: 1000
  patent_id        ipc          cpc
0  14905169    A61F208      A61F208
1  14994322    D21G900    D21G90063
2  14994587    B60N248    B60N24885
3  14903072  H01L21673  H01L2167389
4  14993448    G02B630      G02B630


In [26]:
import numpy as np
import random

random.seed(2024)
np.random.seed(2024)

def create_validation_pairs(df, n_positive=1000, n_negative=1000):

    df = df.reset_index(drop=True)

    positive_pairs = []
    negative_pairs = []

    # -------------------------
    # Positive pairs
    # Same IPC OR same CPC
    # -------------------------
    groups = {}

    for i, row in df.iterrows():

        ipc = str(row["ipc"]).strip()
        cpc = str(row["cpc"]).strip()

        if ipc and ipc != "nan":
            groups.setdefault(ipc, []).append(i)

        if cpc and cpc != "nan":
            groups.setdefault(cpc, []).append(i)

    for indices in groups.values():

        indices = list(set(indices))

        if len(indices) < 2:
            continue

        random.shuffle(indices)

        for j in range(len(indices) - 1):

            a = indices[j]
            b = indices[j + 1]

            positive_pairs.append((a, b, 1))

            if len(positive_pairs) >= n_positive:
                break

        if len(positive_pairs) >= n_positive:
            break

    # -------------------------
    # Negative pairs
    # Different IPC AND CPC
    # -------------------------
    while len(negative_pairs) < n_negative:

        a, b = np.random.choice(
            len(df),
            2,
            replace=False
        )

        ipc_a = str(df.iloc[a]["ipc"])
        ipc_b = str(df.iloc[b]["ipc"])

        cpc_a = str(df.iloc[a]["cpc"])
        cpc_b = str(df.iloc[b]["cpc"])

        if ipc_a != ipc_b and cpc_a != cpc_b:

            negative_pairs.append((a, b, 0))

    pairs = positive_pairs + negative_pairs

    random.shuffle(pairs)

    return pairs


val_pairs = create_validation_pairs(
    val_df,
    n_positive=1000,
    n_negative=1000
)

print("Total pairs:", len(val_pairs))
print("Positive:", sum(p[2] == 1 for p in val_pairs))
print("Negative:", sum(p[2] == 0 for p in val_pairs))

Total pairs: 1235
Positive: 235
Negative: 1000


In [27]:
def get_model_text(row):
    return (
        "Title: " + str(row["title"]) +
        " Abstract: " + str(row["abstract"]) +
        " Claims: " + str(row["claims"])
    )

In [28]:
# ============================================================
# VALIDATION SENTENCE PAIRS
# ============================================================

val_sentences1 = []
val_sentences2 = []
val_labels = []


for a, b, label in val_pairs:

    val_sentences1.append(
        get_model_text(val_df.iloc[a])
    )

    val_sentences2.append(
        get_model_text(val_df.iloc[b])
    )

    val_labels.append(int(label))


print("Validation sentence pairs:", len(val_labels))
print("Positive:", sum(val_labels))
print("Negative:", len(val_labels) - sum(val_labels))

Validation sentence pairs: 1235
Positive: 235
Negative: 1000


In [29]:
# ============================================================
# VALIDATION EVALUATOR
# ============================================================

from sentence_transformers.evaluation import BinaryClassificationEvaluator

val_evaluator = BinaryClassificationEvaluator(
    sentences1=val_sentences1,
    sentences2=val_sentences2,
    labels=val_labels,
    name="patent_validation",
    batch_size=16,
    show_progress_bar=True
)

print("Validation evaluator ready.")

Validation evaluator ready.


/tmp/ipykernel_49/1901795067.py:5: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import BinaryClassificationEvaluator


In [30]:
# ============================================================
# FREEZE ALL EXCEPT LAST 3 TRANSFORMER LAYERS
# ============================================================

transformer = model[0]
auto_model = transformer.auto_model

# Freeze everything
for param in auto_model.parameters():
    param.requires_grad = False

# Unfreeze last 3 layers
for layer in auto_model.encoder.layer[-3:]:
    for param in layer.parameters():
        param.requires_grad = True

# Pooler
if hasattr(auto_model, "pooler") and auto_model.pooler is not None:
    for param in auto_model.pooler.parameters():
        param.requires_grad = True

print("Only last 3 transformer layers are trainable.")

Only last 3 transformer layers are trainable.


In [31]:
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)
print(
    "Trainable percentage:",
    round(100 * trainable_params / total_params, 2),
    "%"
)

Trainable parameters: 21854208
Total parameters: 109486464
Trainable percentage: 19.96 %


In [32]:
model.max_seq_length = 256

print("Max sequence length:", model.max_seq_length)

Max sequence length: 256


In [33]:
from sentence_transformers import losses
import os
import json

# ============================================================
# 1. LOSS FUNCTION
# ============================================================

train_loss = losses.CosineSimilarityLoss(
    model=model
)


# ============================================================
# 2. KAGGLE OUTPUT PATHS
# ============================================================

OUTPUT_PATH = "/kaggle/working/patentsberta_best_model"

CHECKPOINT_PATH = "/kaggle/working/patent_checkpoints"

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)


# ============================================================
# 3. CHECKPOINT CALLBACK
# ============================================================

training_history = []


def save_checkpoint(score, epoch, steps):

    epoch_number = epoch + 1

    checkpoint_path = os.path.join(
        CHECKPOINT_PATH,
        f"epoch_{epoch_number}"
    )

    os.makedirs(checkpoint_path, exist_ok=True)

    # Save model
    model.save(checkpoint_path)

    # Save checkpoint information
    checkpoint_info = {
        "epoch": epoch_number,
        "steps": steps,
        "validation_score": float(score)
    }

    with open(
        os.path.join(checkpoint_path, "checkpoint_info.json"),
        "w"
    ) as f:
        json.dump(checkpoint_info, f, indent=4)

    training_history.append(checkpoint_info)

    print(f"\n✅ Checkpoint saved: Epoch {epoch_number}")
    print(f"📁 {checkpoint_path}")


# ============================================================
# 4. TRAIN MODEL
# ============================================================

model.fit(
    train_objectives=[
        (train_loader, train_loss)
    ],

    # Validation
    evaluator=val_evaluator,

    # Training
    epochs=10,

    # 0 = evaluate at the end of every epoch
    evaluation_steps=0,

    # Warmup
    warmup_steps=500,

    # Learning rate
    optimizer_params={
        "lr": 5e-6
    },

    # Best model location
    output_path=OUTPUT_PATH,

    # Automatically keep the best validation model
    save_best_model=True,

    # Save checkpoint after every epoch
    callback=save_checkpoint,

    # Progress bar
    show_progress_bar=True
)


# ============================================================
# 5. SAVE TRAINING HISTORY
# ============================================================

with open(
    "/kaggle/working/training_history.json",
    "w"
) as f:
    json.dump(
        training_history,
        f,
        indent=4
    )


print("\n========================================")
print("🎉 TRAINING COMPLETED")
print("========================================")

print("\n Best model:")
print(OUTPUT_PATH)

print("\n Epoch checkpoints:")
print(CHECKPOINT_PATH)

print("\n Training history:")
print("/kaggle/working/training_history.json")

Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sentence_transformers/util/tensor.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  a = torch.tensor(a)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Checkpoint saved: Epoch 2.0
📁 /kaggle/working/patent_checkpoints/epoch_2.0


... (23 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Checkpoint saved: Epoch 11.0
📁 /kaggle/working/patent_checkpoints/epoch_11.0

🎉 TRAINING COMPLETED

 Best model:
/kaggle/working/patentsberta_best_model

 Epoch checkpoints:
/kaggle/working/patent_checkpoints

 Training history:
/kaggle/working/training_history.json


In [34]:
# ============================================================
# LOAD BEST VALIDATION MODEL
# ============================================================

from sentence_transformers import SentenceTransformer

best_model = SentenceTransformer(
    OUTPUT_PATH
)

print("Best model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Best model loaded.


In [35]:
# ============================================================
# FINAL VALIDATION EVALUATION
# ============================================================

validation_result = val_evaluator(
    best_model
)

print(validation_result)

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

{'patent_validation_cosine_accuracy': 0.8712550607287449, 'patent_validation_cosine_accuracy_threshold': 0.6820555925369263, 'patent_validation_cosine_f1': 0.6742424242424242, 'patent_validation_cosine_f1_threshold': 0.42118895053863525, 'patent_validation_cosine_precision': 0.6075085324232082, 'patent_validation_cosine_recall': 0.7574468085106383, 'patent_validation_cosine_ap': 0.7084464541978771, 'patent_validation_cosine_mcc': 0.5928049829889634}


In [36]:
train_loss = losses.CosineSimilarityLoss(
    model=model
)


model.fit(
    train_objectives=[
        (train_loader, train_loss)
    ],

    epochs=2,

    warmup_steps=500,

    optimizer_params={
        "lr": 5e-6
    },

    show_progress_bar=True
)

Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Step,Training Loss
500,0.060878
1000,0.058465
1500,0.056956
2000,0.051549
2500,0.054286
3000,0.050698
3500,0.052157
4000,0.055646


In [37]:
del train_loader
del train_dataset
del train_pairs

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Training memory cleared.")

Training memory cleared.


In [38]:
texts = df["model_text"].tolist()

finetuned_embeddings = model.encode(
    texts,
    batch_size=2,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

finetuned_embeddings = (
    finetuned_embeddings
    .astype(np.float32)
)

del texts

gc.collect()

print(
    "Shape:",
    finetuned_embeddings.shape
)

Batches:   0%|          | 0/5000 [00:00<?, ?it/s]

Shape: (10000, 768)


In [39]:
from sentence_transformers import SentenceTransformer

PRETRAINED_MODEL = "AI-Growth-Lab/PatentSBERTa"

pretrained_model = SentenceTransformer(PRETRAINED_MODEL)

print("Pretrained model loaded")

#pre-trained embeddings
# Generate pretrained embeddings for the complete dataset

all_texts = df["model_text"].tolist()

pretrained_embeddings = pretrained_model.encode(
    all_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Pretrained embeddings shape:", pretrained_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Pretrained model loaded


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Pretrained embeddings shape: (10000, 768)


In [40]:
#TF-IDF Similarity
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=30000,
    stop_words="english",
    ngram_range=(1, 2),

    min_df=2,
    dtype=np.float32
)

In [41]:
train_texts = (
    df.iloc[train_idx]
    ["model_text"]
    .tolist()
)

tfidf_vectorizer.fit(
    train_texts
)

del train_texts

gc.collect()

46

In [42]:
tfidf_matrix = (
    tfidf_vectorizer
    .transform(df["model_text"])
)

print(
    "TF-IDF:",
    tfidf_matrix.shape
)

TF-IDF: (10000, 30000)


In [43]:
claim_vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    dtype=np.float32
)


train_claims = (
    df.iloc[train_idx]
    ["claims"]
    .tolist()
)

claim_vectorizer.fit(
    train_claims
)

del train_claims

gc.collect()


claim_tfidf_matrix = (
    claim_vectorizer
    .transform(df["claims"])
)


print(
    "Claim TF-IDF:",
    claim_tfidf_matrix.shape
)

Claim TF-IDF: (10000, 20000)


In [44]:
#keyword-Similarity
from collections import Counter


def extract_keywords(
    text,
    top_n=50
):

    if not text:
        return set()

    words = re.findall(
        r"\b[a-zA-Z]{3,}\b",
        str(text).lower()
    )

    stopwords = {
        "the", "and", "for",
        "with", "that", "this",
        "from", "are", "was",
        "were", "has", "have",
        "into", "using", "used",
        "such", "which", "where",
        "when", "than", "then",
        "their", "there", "these",
        "those", "method", "system",
        "device", "apparatus"
    }

    words = [
        w
        for w in words
        if w not in stopwords
    ]

    counts = Counter(words)

    return set(
        word
        for word, count
        in counts.most_common(top_n)
    )


def keyword_jaccard(
    text1,
    text2
):

    k1 = extract_keywords(text1)
    k2 = extract_keywords(text2)

    if not k1 or not k2:
        return 0.0

    intersection = len(
        k1 & k2
    )

    union = len(
        k1 | k2
    )

    return (
        intersection / union
        if union > 0
        else 0.0
    )

In [45]:
#Hierrarchical IPC/CPC
def split_codes(value):

    if value is None:
        return []

    if isinstance(value, list):

        return [
            normalize_classification(x)
            for x in value
            if normalize_classification(x)
        ]

    value = str(value)

    parts = re.split(
        r"[;,|]+",
        value
    )

    return [
        normalize_classification(x)
        for x in parts
        if normalize_classification(x)
    ]

In [46]:

def hierarchical_code_similarity(
    code1,
    code2
):

    code1 = normalize_classification(code1)
    code2 = normalize_classification(code2)

    if not code1 or not code2:
        return 0.0


    # Section
    if code1[0] != code2[0]:
        return 0.0

    score = 0.10


    # Class
    if (
        len(code1) >= 3
        and
        len(code2) >= 3
    ):

        if code1[:3] == code2[:3]:

            score = 0.25

        else:

            return score


    # Subclass
    if (
        len(code1) >= 4
        and
        len(code2) >= 4
    ):

        if code1[:4] == code2[:4]:

            score = 0.45

        else:

            return score


    suffix1 = code1[4:]
    suffix2 = code2[4:]


    if not suffix1 or not suffix2:
        return score


    # Numeric level 1
    if suffix1[0] == suffix2[0]:

        score = 0.60

    else:

        return score


    # Numeric level 2
    if (
        len(suffix1) >= 2
        and
        len(suffix2) >= 2
    ):

        if suffix1[:2] == suffix2[:2]:

            score = 0.75

        else:

            return score


    # Numeric level 3
    if (
        len(suffix1) >= 3
        and
        len(suffix2) >= 3
    ):

        if suffix1[:3] == suffix2[:3]:

            score = 0.85

        else:

            return score


    # Numeric level 4
    if (
        len(suffix1) >= 4
        and
        len(suffix2) >= 4
    ):

        if suffix1[:4] == suffix2[:4]:

            score = 0.92

        else:

            return score


    # Exact match
    if code1 == code2:

        score = 1.0


    return score

In [47]:
def hierarchical_multi_code_similarity(
    codes1,
    codes2
):

    codes1 = split_codes(codes1)
    codes2 = split_codes(codes2)

    if not codes1 or not codes2:
        return 0.0

    max_score = 0.0

    for c1 in codes1:

        for c2 in codes2:

            score = hierarchical_code_similarity(
                c1,
                c2
            )

            max_score = max(
                max_score,
                score
            )

    return max_score

In [48]:
#HyBrid
HYBRID_WEIGHTS = np.array(
    [
        0.3333,  # pre-trained PatentSBERTa
        0.1111,  # TF-IDF
        0.2222,  # Claim TF-IDF
        0.0556,  # Keyword
        0.1111,  # Hierarchical IPC
        0.1667   # Hierarchical CPC
    ],
    dtype=np.float32
)

print("Weight sum:", HYBRID_WEIGHTS.sum())

Weight sum: 1.0


In [49]:
#Patent 0 as  Query
query_idx = 0

query_patent = df.iloc[query_idx]

print("QUERY PATENT")
print("==============================")
print("Index    :", query_idx)
print("Patent ID:", query_patent["patent_id"])
print("Title    :", query_patent["title"])

QUERY PATENT
Index    : 0
Patent ID: 14991751
Title    : LIQUID EJECTING APPARATUS AND METHOD OF DETECTION


In [50]:
query_embedding = (
    pretrained_embeddings[
        query_idx
    ]
)


candidate_scores = np.dot(
    pretrained_embeddings,
    query_embedding
)

In [51]:
candidate_scores[
    query_idx
] = -1

In [52]:
TOP_CANDIDATES = 100


candidate_indices = np.argpartition(
    candidate_scores,
    -TOP_CANDIDATES
)[-TOP_CANDIDATES:]

In [53]:
candidate_indices = candidate_indices[
    np.argsort(
        candidate_scores[
            candidate_indices
        ]
    )[::-1]
]

In [54]:
# ============================================================
# CALCULATE 6 SIMILARITY FEATURES FOR ALL 100 CANDIDATES
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

results = []

query_idx = 0
query = df.iloc[query_idx]

for candidate_idx in tqdm(
    candidate_indices,
    desc="Calculating similarities"
):

    candidate = df.iloc[candidate_idx]

    # --------------------------------------------------------
    # 1. Fine-tuned PatentSBERTa
    # --------------------------------------------------------

    finetune_sim = float(
        np.dot(
            finetuned_embeddings[query_idx],
            finetuned_embeddings[candidate_idx]
        )
    )



    # --------------------------------------------------------
    # 2. Full-text TF-IDF
    # --------------------------------------------------------

    tfidf_sim = float(
        cosine_similarity(
            tfidf_matrix[query_idx],
            tfidf_matrix[candidate_idx]
        )[0, 0]
    )

    # --------------------------------------------------------
    # 3. Claim TF-IDF
    # --------------------------------------------------------

    claim_sim = float(
        cosine_similarity(
            claim_tfidf_matrix[query_idx],
            claim_tfidf_matrix[candidate_idx]
        )[0, 0]
    )

    # --------------------------------------------------------
    # 4. Keyword Jaccard
    # --------------------------------------------------------

    keyword_sim = float(
        keyword_jaccard(
            query["model_text"],
            candidate["model_text"]
        )
    )

    # --------------------------------------------------------
    # 5. Hierarchical IPC
    # --------------------------------------------------------

    ipc_sim = float(
        hierarchical_multi_code_similarity(
            query["ipc"],
            candidate["ipc"]
        )
    )

    # --------------------------------------------------------
    # 6. Hierarchical CPC
    # --------------------------------------------------------

    cpc_sim = float(
        hierarchical_multi_code_similarity(
            query["cpc"],
            candidate["cpc"]
        )
    )

    # --------------------------------------------------------
    # Feature vector
    # --------------------------------------------------------

    features = np.array(
        [
            finetune_sim,
            tfidf_sim,
            claim_sim,
            keyword_sim,
            ipc_sim,
            cpc_sim
        ],
        dtype=np.float32
    )

    # --------------------------------------------------------
    # Hybrid score
    # --------------------------------------------------------

    hybrid_score = float(
        np.dot(
            features,
            HYBRID_WEIGHTS
        )
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # This MUST be inside the for loop
    # --------------------------------------------------------

    results.append(
        {
            "patent_index": int(candidate_idx),

            "patent_id": candidate["patent_id"],

            "title": candidate["title"],

            "similarity_score": hybrid_score,

            "finetuned_similarity": finetune_sim,

            "tfidf_similarity": tfidf_sim,

            "claim_similarity": claim_sim,

            "keyword_similarity": keyword_sim,

            "ipc_similarity": ipc_sim,

            "cpc_similarity": cpc_sim
        }
    )


print("Number of results:", len(results))

Calculating similarities:   0%|          | 0/100 [00:00<?, ?it/s]

Number of results: 100


In [55]:
results_df = pd.DataFrame(
    results
)


results_df = results_df.sort_values(
    by="similarity_score",
    ascending=False
)


top_10 = (
    results_df
    .head(10)
    .reset_index(drop=True)
)


top_10.insert(
    0,
    "rank",
    range(
        1,
        len(top_10) + 1
    )
)

In [56]:
print(df.shape)
print(df["patent_id"].head())
print(df["patent_id"].nunique())

(10000, 9)
0    14991751
1    14995025
2    14904988
3    14987351
4    14772934
Name: patent_id, dtype: object
10000


In [57]:
print("Number of patents in df:", len(df))
print("Number of candidate indices:", len(candidate_indices))
print("Number of results:", len(results))

print("\nCandidate indices:")
print(candidate_indices[:20])

print("\nResults shape:")
print(results_df.shape)

Number of patents in df: 10000
Number of candidate indices: 100
Number of results: 100

Candidate indices:
[4658 8746 2741 2434 2208 2210 6921 1565 9321 9636 8219  202 9583 6429
 1668 2191 2406 2260  724 6950]

Results shape:
(100, 10)


In [58]:
top_10[
    [
        "rank",
        "patent_index",
        "patent_id",
        "similarity_score"
    ]
]

,rank,patent_index,patent_id,similarity_score
0,1,4658,14996671,0.627935
1,2,6921,15000255,0.595032
2,3,8746,14997840,0.593901
3,4,9636,15000527,0.558205
4,5,2210,14995875,0.551297
5,6,2208,14993849,0.537443
6,7,1565,14992610,0.492968
7,8,6736,14988379,0.492890
8,9,2406,14993481,0.489414
9,10,4738,14991014,0.480515


In [59]:
top_10[
    [
        "rank",
        "patent_id",
        "similarity_score",
        "finetuned_similarity",
        "tfidf_similarity",
        "claim_similarity",
        "keyword_similarity",
        "ipc_similarity",
        "cpc_similarity"
    ]
]

,rank,patent_id,similarity_score,finetuned_similarity,tfidf_similarity,claim_similarity,keyword_similarity,ipc_similarity,cpc_similarity
0,1,14996671,0.627935,0.844066,0.503384,0.510404,0.190476,0.60,0.60
1,2,15000255,0.595032,0.847853,0.275838,0.293154,0.149425,0.75,0.75
2,3,14997840,0.593901,0.886497,0.329428,0.394105,0.136364,0.60,0.60
3,4,15000527,0.558205,0.819090,0.200271,0.414433,0.075269,0.60,0.60
4,5,14995875,0.551297,0.817490,0.209247,0.378334,0.086957,0.60,0.60
5,6,14993849,0.537443,0.836734,0.408151,0.359569,0.149425,0.45,0.45
6,7,14992610,0.492968,0.813391,0.138414,0.145027,0.136364,0.60,0.60
7,8,14988379,0.492890,0.858604,0.105190,0.111620,0.063830,0.60,0.60
8,9,14993481,0.489414,0.878861,0.069523,0.077637,0.086957,0.60,0.60
9,10,14991014,0.480515,0.816076,0.102018,0.106349,0.123596,0.60,0.60


In [60]:
def is_relevant(
    query_row,
    candidate_row
):

    query_ipc = subclass_code(
        query_row["ipc"]
    )

    candidate_ipc = subclass_code(
        candidate_row["ipc"]
    )

    query_cpc = subclass_code(
        query_row["cpc"]
    )

    candidate_cpc = subclass_code(
        candidate_row["cpc"]
    )

    return (
        query_ipc == candidate_ipc
        or
        query_cpc == candidate_cpc
    )

In [61]:
relevance = []


for _, row in top_10.iterrows():

    candidate = df.iloc[
        int(row["patent_index"])
    ]

    relevance.append(
        is_relevant(
            query_patent,
            candidate
        )
    )


top_10["relevant"] = relevance


precision_at_10 = (
    top_10["relevant"].sum()
    / 10
)


print(
    "Precision@10:",
    precision_at_10
)

Precision@10: 1.0


In [62]:
hit_at_10 = int(
    top_10["relevant"].sum() > 0
)

print(
    "Hit@10:",
    hit_at_10
)

Hit@10: 1


In [63]:
top_10[
    [
        "rank",
        "patent_id",
        "similarity_score",
        "ipc_similarity",
        "cpc_similarity",
        "relevant"
    ]
]

,rank,patent_id,similarity_score,ipc_similarity,cpc_similarity,relevant
0,1,14996671,0.627935,0.60,0.60,True
1,2,15000255,0.595032,0.75,0.75,True
2,3,14997840,0.593901,0.60,0.60,True
3,4,15000527,0.558205,0.60,0.60,True
4,5,14995875,0.551297,0.60,0.60,True
5,6,14993849,0.537443,0.45,0.45,True
6,7,14992610,0.492968,0.60,0.60,True
7,8,14988379,0.492890,0.60,0.60,True
8,9,14993481,0.489414,0.60,0.60,True
9,10,14991014,0.480515,0.60,0.60,True


In [64]:
query = df.iloc[0]

print("QUERY PATENT")
print("----------------")
print("Patent ID:", query["patent_id"])
print("IPC:", query["ipc"])
print("CPC:", query["cpc"])

print("\nTOP 10 CLASSIFICATIONS")
print("----------------------------")

for _, row in top_10.iterrows():

    idx = int(row["patent_index"])

    candidate = df.iloc[idx]

    print(
        "\nRank:", row["rank"],
        "\nPatent:", candidate["patent_id"],
        "\nIPC:", candidate["ipc"],
        "\nCPC:", candidate["cpc"],
        "\nSimilarity:", row["similarity_score"]
    )

QUERY PATENT
----------------
Patent ID: 14991751
IPC: B41J25308
CPC: B41J25308

TOP 10 CLASSIFICATIONS
----------------------------

Rank: 1 
Patent: 14996671 
IPC: B41J2045 
CPC: B41J204563 
Similarity: 0.6279352903366089

Rank: 2 
Patent: 15000255 
IPC: B41J2500 
CPC: B41J25006 
Similarity: 0.5950319170951843

Rank: 3 
Patent: 14997840 
IPC: B41J2175 
CPC: B41J217596 
Similarity: 0.5939009785652161

Rank: 4 
Patent: 15000527 
IPC: B41J2165 
CPC: B41J216505 
Similarity: 0.5582048892974854

Rank: 5 
Patent: 14995875 
IPC: B41J2165 
CPC: B41J216505 
Similarity: 0.5512974858283997

Rank: 6 
Patent: 14993849 
IPC: B41J1104 
CPC: B41J1104 
Similarity: 0.5374431610107422

Rank: 7 
Patent: 14992610 
IPC: B41J2175 
CPC: B41J2175 
Similarity: 0.4929678738117218

Rank: 8 
Patent: 14988379 
IPC: B41J214 
CPC: B41J21433 
Similarity: 0.492890328168869

Rank: 9 
Patent: 14993481 
IPC: B41J29393 
CPC: B41J29393 
Similarity: 0.4894142746925354

Rank: 10 
Patent: 14991014 
IPC: B41J214 
CPC: B41J2142

In [65]:
#Query  patent
# ============================================================
# HAND-CODED QUERY PATENT
# ============================================================

hand_query = {
    "patent_id": "HAND_QUERY_001",

    "title": "Electroplating apparatus with improved article supporting device",

    "abstract": """
    An electroplating apparatus for applying a metallic coating to an article.
    The apparatus includes a supporting device configured to hold and suspend
    the article during an electrolytic coating process. The supporting device
    improves electrical contact and maintains the position of the article
    inside an electroplating bath.
    """,

    "claims": """
    1. An electroplating apparatus comprising a supporting device configured
       to suspend an article during an electrolytic coating process.

    2. The apparatus of claim 1, wherein the supporting device maintains
       electrical contact with the article during electroplating.

    3. The apparatus of claim 1, wherein the article is positioned inside
       an electrolyte bath during the coating process.
    """,

    # Same classification hierarchy as your Patent 0
    "ipc": "C25D1706",
    "cpc": "C25D1706"
}

print("Hand-coded query patent")
print("-----------------------")
print("Patent ID:", hand_query["patent_id"])
print("Title:", hand_query["title"])
print("IPC:", hand_query["ipc"])
print("CPC:", hand_query["cpc"])

Hand-coded query patent
-----------------------
Patent ID: HAND_QUERY_001
Title: Electroplating apparatus with improved article supporting device
IPC: C25D1706
CPC: C25D1706


In [66]:
print(
    "IPC similarity:",
    hierarchical_multi_code_similarity(
        hand_query["ipc"],
        "C25D1706"
    )
)

print(
    "CPC similarity:",
    hierarchical_multi_code_similarity(
        hand_query["cpc"],
        "C25D1706"
    )
)

IPC similarity: 1.0
CPC similarity: 1.0


In [67]:
test_codes = [
    "C25D1706",
    "C25D1704",
    "C25D1710",
    "C25D17",
    "C25D11",
    "C25F1706",
    "B23K2012"
]

for code in test_codes:

    score = hierarchical_code_similarity(
        "C25D1706",
        code
    )

    print(
        "C25D1706 vs",
        code,
        "=>",
        score
    )

C25D1706 vs C25D1706 => 1.0
C25D1706 vs C25D1704 => 0.85
C25D1706 vs C25D1710 => 0.75
C25D1706 vs C25D17 => 0.75
C25D1706 vs C25D11 => 0.6
C25D1706 vs C25F1706 => 0.25
C25D1706 vs B23K2012 => 0.0


In [68]:
matches = df[
    (
        df["ipc"].astype(str).str.contains(
            "C25D1706",
            regex=False,
            na=False
        )
    )
    |
    (
        df["cpc"].astype(str).str.contains(
            "C25D1706",
            regex=False,
            na=False
        )
    )
]

print("Matching patents:", len(matches))

print(
    matches[
        [
            "patent_id",
            "ipc",
            "cpc",
            "title"
        ]
    ].head(20)
)

Matching patents: 1
     patent_id       ipc       cpc  \
4555  14904285  C25D1706  C25D1706   

                                                  title  
4555  SUPPORTING TOOL FOR SUPPORTING CYLINDRICAL PAR...  


In [69]:
#Use the hand-coded patent as the query
hand_query_text = (
    "Title: " +
    hand_query["title"] +

    " Abstract: " +
    hand_query["abstract"] +

    " Claims: " +
    hand_query["claims"]
)

In [70]:
hand_query_embedding = pretrained_model.encode(
    [hand_query_text],
    convert_to_numpy=True,
    normalize_embeddings=True
)[0].astype(np.float32)

print(
    "Embedding shape:",
    hand_query_embedding.shape
)

Embedding shape: (768,)


In [71]:
candidate_scores = np.dot(
    pretrained_embeddings,
    hand_query_embedding
)

TOP_CANDIDATES = 100

candidate_indices = np.argpartition(
    candidate_scores,
    -TOP_CANDIDATES
)[-TOP_CANDIDATES:]

candidate_indices = candidate_indices[
    np.argsort(
        candidate_scores[candidate_indices]
    )[::-1]
]

print(
    "Candidates:",
    len(candidate_indices)
)

Candidates: 100


In [72]:
hand_results = []

# Generate embedding for the hand query using the finetuned model
finetuned_hand_query_embedding = model.encode(
    [hand_query_text],
    convert_to_numpy=True,
    normalize_embeddings=True
)[0].astype(np.float32)

for candidate_idx in tqdm(
    candidate_indices,
    desc="Calculating hand-query similarities"
):

    candidate = df.iloc[candidate_idx]

    # --------------------------------------------------------
    # 1a. Pretrained PatentSBERTa similarity
    # --------------------------------------------------------

    pretrained_sim = float(
        np.dot(
            hand_query_embedding,
            pretrained_embeddings[candidate_idx]
        )
    )

    # --------------------------------------------------------
    # 1b. Fine-tuned PatentSBERTa similarity
    # --------------------------------------------------------

    finetuned_sim = float(
        np.dot(
            finetuned_hand_query_embedding,
            finetuned_embeddings[candidate_idx]
        )
    )

    # --------------------------------------------
    # 2. TF-IDF
    # --------------------------------------------

    hand_tfidf = tfidf_vectorizer.transform(
        [hand_query_text]
    )

    tfidf_sim = float(
        cosine_similarity(
            hand_tfidf,
            tfidf_matrix[candidate_idx]
        )[0, 0]
    )

    # --------------------------------------------
    # 3. Claim TF-IDF
    # --------------------------------------------

    hand_claim_tfidf = claim_vectorizer.transform(
        [hand_query["claims"]]
    )

    claim_sim = float(
        cosine_similarity(
            hand_claim_tfidf,
            claim_tfidf_matrix[candidate_idx]
        )[0, 0]
    )

    # --------------------------------------------
    # 4. Keyword similarity
    # --------------------------------------------

    keyword_sim = float(
        keyword_jaccard(
            hand_query_text,
            candidate["model_text"]
        )
    )

    # --------------------------------------------
    # 5. Hierarchical IPC
    # --------------------------------------------

    ipc_sim = float(
        hierarchical_multi_code_similarity(
            hand_query["ipc"],
            candidate["ipc"]
        )
    )

    # --------------------------------------------
    # 6. Hierarchical CPC
    # --------------------------------------------

    cpc_sim = float(
        hierarchical_multi_code_similarity(
            hand_query["cpc"],
            candidate["cpc"]
        )
    )

    # --------------------------------------------
    # Hybrid
    # --------------------------------------------

    features = np.array(
        [
            finetuned_sim,
            tfidf_sim,
            claim_sim,
            keyword_sim,
            ipc_sim,
            cpc_sim
        ],
        dtype=np.float32
    )

    hybrid_score = float(
        np.dot(
            features,
            HYBRID_WEIGHTS
        )
    )

    hand_results.append(
        {
            "patent_index": int(candidate_idx),
            "patent_id": candidate["patent_id"],
            "title": candidate["title"],

            "similarity_score": hybrid_score,

            "pretrained_similarity": pretrained_sim,
            "finetuned_similarity": finetuned_sim,
            "tfidf_similarity": tfidf_sim,
            "claim_similarity": claim_sim,
            "keyword_similarity": keyword_sim,

            "ipc_similarity": ipc_sim,
            "cpc_similarity": cpc_sim
        }
    )

Calculating hand-query similarities:   0%|          | 0/100 [00:00<?, ?it/s]

In [73]:
hand_results_df = pd.DataFrame(
    hand_results
)

hand_top_10 = (
    hand_results_df
    .sort_values(
        "similarity_score",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

hand_top_10.insert(
    0,
    "rank",
    range(1, 11)
)

In [74]:
hand_top_10[
    [
        "rank",
        "patent_id",
        "similarity_score",
        "finetuned_similarity",
        "tfidf_similarity",
        "claim_similarity",
        "keyword_similarity",
        "ipc_similarity",
        "cpc_similarity"
    ]
]

,rank,patent_id,similarity_score,finetuned_similarity,tfidf_similarity,claim_similarity,keyword_similarity,ipc_similarity,cpc_similarity
0,1,14987170,0.498217,0.917611,0.244153,0.152911,0.112676,0.45,0.45
1,2,14904285,0.418472,0.295183,0.142576,0.094696,0.097222,1.00,1.00
2,3,14903677,0.334417,0.815692,0.145837,0.066642,0.067568,0.10,0.10
3,4,14903201,0.328572,0.770338,0.084111,0.131808,0.097222,0.10,0.10
4,5,14902979,0.318718,0.778031,0.115312,0.064083,0.082192,0.10,0.10
5,6,14986956,0.318364,0.832369,0.089744,0.107184,0.128571,0.00,0.00
6,7,14903824,0.300896,0.835254,0.052534,0.054455,0.082192,0.00,0.00
7,8,15001452,0.298711,0.849932,0.018123,0.032180,0.112676,0.00,0.00
8,9,14903479,0.295531,0.785647,0.064972,0.098498,0.082192,0.00,0.00
9,10,14986935,0.294201,0.803008,0.056434,0.059137,0.128571,0.00,0.00


In [75]:
def get_model_text(row):

    return (
        "Title: " + str(row["title"]) +
        " Abstract: " + str(row["abstract"]) +
        " Claims: " + str(row["claims"])
    )


val_texts = [
    get_model_text(row)
    for _, row in val_df.iterrows()
]

print("Validation texts:", len(val_texts))

Validation texts: 1000


In [76]:
finetuned_model = model

In [77]:
from sentence_transformers import SentenceTransformer

PRETRAINED_MODEL = "AI-Growth-Lab/PatentSBERTa"

pretrained_model = SentenceTransformer(PRETRAINED_MODEL)

print("Pretrained model loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Pretrained model loaded


In [78]:
print("Pretrained model...")

pretrained_embeddings = pretrained_model.encode(
    val_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Fine-tuned model...")

finetuned_embeddings = finetuned_model.encode(
    val_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Pretrained:", pretrained_embeddings.shape)
print("Fine-tuned:", finetuned_embeddings.shape)

Pretrained model...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Fine-tuned model...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Pretrained: (1000, 768)
Fine-tuned: (1000, 768)


In [79]:
pretrained_scores = []
finetuned_scores = []
labels = []

for a, b, label in val_pairs:

    pretrained_scores.append(
        np.dot(
            pretrained_embeddings[a],
            pretrained_embeddings[b]
        )
    )

    finetuned_scores.append(
        np.dot(
            finetuned_embeddings[a],
            finetuned_embeddings[b]
        )
    )

    labels.append(label)


pretrained_scores = np.array(pretrained_scores)
finetuned_scores = np.array(finetuned_scores)
labels = np.array(labels)

In [80]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

pretrained_auc = roc_auc_score(
    labels,
    pretrained_scores
)

finetuned_auc = roc_auc_score(
    labels,
    finetuned_scores
)

pretrained_pr = average_precision_score(
    labels,
    pretrained_scores
)

finetuned_pr = average_precision_score(
    labels,
    finetuned_scores
)

print("================================")
print("MODEL COMPARISON")
print("================================")

print(f"Pretrained ROC-AUC : {pretrained_auc:.4f}")
print(f"Fine-tuned ROC-AUC : {finetuned_auc:.4f}")

print()

print(f"Pretrained PR-AUC  : {pretrained_pr:.4f}")
print(f"Fine-tuned PR-AUC  : {finetuned_pr:.4f}")

MODEL COMPARISON
Pretrained ROC-AUC : 0.8946
Fine-tuned ROC-AUC : 0.8968

Pretrained PR-AUC  : 0.7590
Fine-tuned PR-AUC  : 0.6922


In [81]:
from sklearn.metrics import accuracy_score

# Convert similarity scores into predictions
pretrained_pred = (pretrained_scores >= 0.5).astype(int)
finetuned_pred = (finetuned_scores >= 0.5).astype(int)

# Accuracy
pretrained_accuracy = accuracy_score(
    labels,
    pretrained_pred
) * 100

finetuned_accuracy = accuracy_score(
    labels,
    finetuned_pred
) * 100


print("====================================")
print("MODEL ACCURACY")
print("====================================")

print(f"Pretrained Accuracy : {pretrained_accuracy:.2f}%")
print(f"Fine-tuned Accuracy : {finetuned_accuracy:.2f}%")

MODEL ACCURACY
Pretrained Accuracy : 83.56%
Fine-tuned Accuracy : 85.26%
